# WMG — Artificial Intelligence & Deep Learning
## Hands-on Exercise: Building a RAG Pipeline

**Module:** WM9B7 Artificial Intelligence & Deep Learning  
**Lecturer:** Matheus Torquato  

---

### Overview

In this notebook you will build a complete Retrieval Augmented Generation (RAG) pipeline from scratch.  
By the end of the exercise you will have:

1. Read, chunked, and embedded a set of documents
2. Stored the embedded chunks in a PostgreSQL vector database
3. Queried the vector database to verify retrieval works
4. Made calls to an LLM via the Groq API
5. Built a LangChain RAG chain that retrieves relevant context before generating an answer
6. Applied re-ranking to improve retrieval precision
7. Called the full RAG chain with re-ranked documents

---

### Prerequisites

Before you start, make sure you have:
- Your **Groq API key** (provided separately)
- Access to the **shared PostgreSQL database** (credentials below)
- All required packages installed (run the cell below)

## 0. Setup — Install Dependencies & Configure Credentials

In [ ]:
# Install required packages
# !pip install langchain langchain-community langchain-groq \
#              psycopg2-binary pgvector sentence-transformers \
#              cohere tiktoken python-dotenv

In [ ]:
# --- Credentials ---
# Replace the placeholders with your own values
# Do NOT commit credentials to version control

GROQ_API_KEY      = "your-groq-api-key"
COHERE_API_KEY    = "your-cohere-api-key"   # used for re-ranking in Step 6

PG_HOST     = "your-postgres-host"
PG_PORT     = 5432
PG_DB       = "your-db-name"
PG_USER     = "your-username"
PG_PASSWORD = "your-password"

# Unique table name — use your student ID to avoid collisions
COLLECTION_NAME = "rag_exercise_studentXXX"

In [ ]:
# Imports
# TODO: add all imports here as you build each step

---
## Step 1 — Read, Chunk, and Embed Documents

The first stage of any RAG pipeline is the **offline ingestion path**:  
load raw documents → split them into chunks → convert each chunk into an embedding vector.

### 1.1 Load Documents

We will use LangChain's document loaders to read the provided PDF files from disk.

In [ ]:
# TODO: Load documents from the /data folder using a LangChain document loader
# Hint: PyPDFDirectoryLoader or DirectoryLoader

documents = []  # placeholder
print(f"Loaded {len(documents)} document(s)")

### 1.2 Split Documents into Chunks

Embedding models have token limits and retrieval precision improves with smaller, focused chunks.  
We will use a **fixed-size splitter with overlap** as our baseline strategy.

In [ ]:
# TODO: Split the loaded documents into chunks
# Hint: RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=...)

chunks = []  # placeholder
print(f"Created {len(chunks)} chunk(s)")

# Inspect the first chunk
# print(chunks[0])

### 1.3 Create Embeddings

We will use a sentence-transformer model to convert each chunk into a dense vector.  
This model runs **locally** — no API key required for this step.

In [ ]:
# TODO: Initialise the embedding model
# Hint: HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embedding_model = None  # placeholder

# Quick sanity check — embed a single sentence and inspect the vector
# test_vector = embedding_model.embed_query("What is retrieval augmented generation?")
# print(f"Embedding dimension: {len(test_vector)}")

---
## Step 2 — Store Embeddings in PostgreSQL (pgvector)

We will connect to the shared PostgreSQL database and store each chunk and its embedding vector.  
PostgreSQL with the **pgvector** extension acts as our vector store.

> ⚠️ Make sure your `COLLECTION_NAME` is unique (use your student ID) to avoid overwriting a classmate's data.

In [ ]:
# TODO: Build the connection string and create the PGVector vector store
# Hint: PGVector.from_documents(documents=chunks, embedding=embedding_model, ...)

CONNECTION_STRING = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"

vector_store = None  # placeholder
print("Vector store created successfully")

---
## Step 3 — Query the Vector Database

Before building the full RAG chain, let's verify the vector store works by running a standalone similarity search.

This step tests the **retrieval component in isolation**.

In [ ]:
# TODO: Run a similarity search against the vector store
# Hint: vector_store.similarity_search(query, k=5)

test_query = "What is the main topic of these documents?"

retrieved_docs = []  # placeholder

# Print the content and metadata of each retrieved chunk
for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:300])
    print(doc.metadata)
    print()

---
## Step 4 — Connect to the LLM via Groq

Groq provides fast inference for open-weight models (LLaMA, Mixtral, etc.) via a simple API.  
We will verify the connection works before wiring it into the RAG chain.

Available models on Groq: `llama3-8b-8192`, `llama3-70b-8192`, `mixtral-8x7b-32768`

In [ ]:
# TODO: Initialise the Groq LLM via LangChain
# Hint: ChatGroq(api_key=GROQ_API_KEY, model_name="llama3-8b-8192")

llm = None  # placeholder

# Quick test — call the LLM directly with a simple prompt
# response = llm.invoke("In one sentence, what is RAG?")
# print(response.content)

---
## Step 5 — Build the RAG LangChain Chain

Now we assemble the full pipeline:

```
User Question
     ↓
Embed question
     ↓
Similarity search → top-k chunks
     ↓
Build prompt (question + context)
     ↓
LLM → Answer
```

### 5.1 Define the Prompt Template

In [ ]:
# TODO: Define a prompt template that instructs the LLM to answer
# using only the provided context
# Hint: ChatPromptTemplate.from_template(...)

SYSTEM_PROMPT = """
You are a helpful assistant. Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {question}
"""

prompt_template = None  # placeholder

### 5.2 Assemble the Chain

In [ ]:
# TODO: Build the RAG chain using LangChain Expression Language (LCEL)
# Components: retriever | prompt | llm | output_parser
# Hint: vector_store.as_retriever(search_kwargs={"k": 10})

retriever  = None  # placeholder
rag_chain  = None  # placeholder

In [ ]:
# Test the basic RAG chain (before re-ranking)
question = "YOUR TEST QUESTION HERE"

# response = rag_chain.invoke(question)
# print(response)

---
## Step 6 — Re-ranking

The vector search retrieves a broad set of candidates optimised for **recall**.  
Re-ranking applies a more accurate cross-encoder model to re-order them by true relevance,  
giving us a smaller, higher-precision set to pass to the LLM.

```
Vector search → top 10 candidates
       ↓
Re-ranker (cross-encoder)
       ↓
Top 3 re-ranked documents → LLM
```

We will use the **Cohere Rerank API** for this step.

In [ ]:
# TODO: Retrieve a larger candidate set from the vector store
# then re-rank with Cohere and keep the top N results

# Step 6a — retrieve a broad candidate set
candidate_docs = []  # placeholder: vector_store.similarity_search(question, k=10)

# Step 6b — re-rank using Cohere
# Hint: import cohere; co = cohere.Client(COHERE_API_KEY)
#       co.rerank(query=question, documents=[...], top_n=3, model="rerank-english-v3.0")

reranked_docs = []  # placeholder

print(f"Candidates before re-ranking : {len(candidate_docs)}")
print(f"Documents after re-ranking   : {len(reranked_docs)}")

In [ ]:
# Inspect the re-ranked results and their relevance scores
for i, doc in enumerate(reranked_docs):
    print(f"--- Re-ranked Result {i+1} ---")
    # print(doc.page_content[:300])
    # print(f"Relevance score: {doc.metadata.get('relevance_score')}")
    print()

---
## Step 7 — Full RAG Chain with Re-ranked Documents

Finally, we call the LLM using the re-ranked, high-precision context instead of the raw retrieval results.

Compare the answer quality between Step 5 (no re-ranking) and Step 7 (with re-ranking).

In [ ]:
# TODO: Format the re-ranked documents into a single context string
# and invoke the LLM with the enriched prompt

def format_docs(docs):
    # TODO: join document page_content into a single string
    return ""  # placeholder

context  = format_docs(reranked_docs)
response = None  # placeholder: llm.invoke(prompt_template.format(context=context, question=question))

print("=== Final Answer ===")
# print(response.content)

---
## Wrap-up & Reflection

Congratulations — you have built a complete RAG pipeline! 🎉

Before finishing, take a few minutes to reflect on the following questions:

**Retrieval**
- How did changing `chunk_size` affect the quality of retrieved documents?
- Did the re-ranker surface different documents compared to the raw vector search?

**Generation**
- Did the LLM ever generate an answer that was not supported by the retrieved context?
- How did the answer change between Step 5 (no re-ranking) and Step 7 (re-ranked)?

**Stretch goals** *(optional)*
- [ ] Try a different chunk size and overlap — does retrieval precision improve?
- [ ] Implement multi-query retrieval: generate 3 variants of your question and merge the results
- [ ] Add a metadata filter to restrict retrieval to a specific document or date range
- [ ] Evaluate your pipeline with RAGAS: measure faithfulness and answer relevance scores